In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

##### Load dữ liệu đã làm sạch

In [24]:
df = pd.read_csv('data\\cleaned_data.csv', parse_dates=['time'], index_col='time')
df = df.sort_index()

rename_mapping = {
    'pm10 (μg/m³)': 'PM10',
    'pm2_5 (μg/m³)': 'PM2.5',
    'carbon_monoxide (μg/m³)': 'Carbon Monoxide',
    'nitrogen_dioxide (μg/m³)': 'Nitrogen Dioxide',
    'sulphur_dioxide (μg/m³)': 'Sulphur Dioxide',
    'ozone (μg/m³)': 'Ozone',
    'uv_index_clear_sky ()': 'UV Index Clear Sky',
    'uv_index ()': 'UV Index',
    'dust (μg/m³)': 'Dust',
    'aerosol_optical_depth ()': 'Aerosol Optical Depth'
}
df = df.rename(columns=rename_mapping)
df

,day,month,year,hour,PM10,PM2.5,Carbon Monoxide,Nitrogen Dioxide,Sulphur Dioxide,Ozone,Aerosol Optical Depth,Dust,UV Index,UV Index Clear Sky
time,,,,,,,,,,,,,,
2022-08-04 07:00:00,4,8,2022,7,11.8,8.3,230.0,3.1,1.0,32.0,0.08,0.0,1.10,1.20
2022-08-04 08:00:00,4,8,2022,8,10.2,7.1,216.0,2.6,0.9,37.0,0.08,0.0,3.25,3.55
2022-08-04 09:00:00,4,8,2022,9,9.3,6.5,195.0,1.8,0.8,43.0,0.08,0.0,6.10,6.95
2022-08-04 10:00:00,4,8,2022,10,9.7,6.8,174.0,1.1,0.8,51.0,0.08,0.0,7.85,10.45
2022-08-04 11:00:00,4,8,2022,11,11.0,7.7,168.0,0.9,0.8,55.0,0.08,0.0,9.55,12.80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-26 19:00:00,26,7,2026,19,13.3,11.2,291.0,4.1,2.6,87.0,0.24,2.0,0.00,0.00
2026-07-26 20:00:00,26,7,2026,20,15.2,12.9,320.0,5.0,3.0,76.0,0.22,2.0,0.00,0.00
2026-07-26 21:00:00,26,7,2026,21,15.7,13.8,343.0,6.1,3.6,61.0,0.20,3.0,0.00,0.00


##### Loại bỏ feature không liên quan (theo kết luận từ 02_eda.ipynb)

In [25]:
# Dust, UV Index, UV Index Clear Sky gần như không tương quan với PM2.5 ở mọi phép đo đã thử trong EDA
# (Correlation Heatmap |r| < 0.1, Cross-Correlation CCF < 0.16 ở mọi lag) -> loại bỏ để giảm nhiễu/chiều dữ liệu
drop_cols = ['Dust', 'UV Index', 'UV Index Clear Sky']
df = df.drop(columns=drop_cols)
print(f"Các cột còn lại: {df.columns.tolist()}")

Các cột còn lại: ['day', 'month', 'year', 'hour', 'PM10', 'PM2.5', 'Carbon Monoxide', 'Nitrogen Dioxide', 'Sulphur Dioxide', 'Ozone', 'Aerosol Optical Depth']


##### Bước 3: Feature Engineering — Calendar Cyclical Features

In [26]:
# Mã hóa hour/month dạng sin/cos để tránh điểm gãy giả tạo (23h và 0h phải "gần nhau", không phải "cách xa 23")
# Bỏ day/year: day không có chu kỳ vật lý rõ ràng với PM2.5, year là số tuyệt đối không generalize được cho
# thời điểm tương lai -> cả hai không phù hợp làm input model.
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df = df.drop(columns=['day', 'month', 'year', 'hour'])
df.head()

,PM10,PM2.5,Carbon Monoxide,Nitrogen Dioxide,Sulphur Dioxide,Ozone,Aerosol Optical Depth,hour_sin,hour_cos,month_sin,month_cos
time,,,,,,,,,,,
2022-08-04 07:00:00,11.8,8.3,230.0,3.1,1.0,32.0,0.08,0.965926,-0.258819,-0.866025,-0.5
2022-08-04 08:00:00,10.2,7.1,216.0,2.6,0.9,37.0,0.08,0.866025,-0.500000,-0.866025,-0.5
2022-08-04 09:00:00,9.3,6.5,195.0,1.8,0.8,43.0,0.08,0.707107,-0.707107,-0.866025,-0.5
2022-08-04 10:00:00,9.7,6.8,174.0,1.1,0.8,51.0,0.08,0.500000,-0.866025,-0.866025,-0.5
2022-08-04 11:00:00,11.0,7.7,168.0,0.9,0.8,55.0,0.08,0.258819,-0.965926,-0.866025,-0.5


##### Bước 4: Train / Validation / Test Split (theo thời gian)

In [27]:
# Chia theo mốc thời gian (KHÔNG shuffle/random) để tránh leakage: 70% train, 15% validation, 15% test
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

print(f"Train: {train_df.index.min()} -> {train_df.index.max()}  ({len(train_df)} dòng)")
print(f"Val:   {val_df.index.min()} -> {val_df.index.max()}  ({len(val_df)} dòng)")
print(f"Test:  {test_df.index.min()} -> {test_df.index.max()}  ({len(test_df)} dòng)")

Train: 2022-08-04 07:00:00 -> 2025-05-17 03:00:00  (24405 dòng)
Val:   2025-05-17 04:00:00 -> 2025-12-21 01:00:00  (5230 dòng)
Test:  2025-12-21 02:00:00 -> 2026-07-26 23:00:00  (5230 dòng)


##### Bước 5: Feature Scaling

In [28]:
# Chỉ fit scaler trên tập TRAIN để tránh leak thống kê của val/test vào lúc train
feature_cols = df.columns.tolist()
pm25_col_idx = feature_cols.index('PM2.5')  # lưu vị trí cột PM2.5 để inverse_transform khi đánh giá model sau này

scaler = MinMaxScaler()
train_scaled = pd.DataFrame(scaler.fit_transform(train_df), columns=feature_cols, index=train_df.index)
val_scaled = pd.DataFrame(scaler.transform(val_df), columns=feature_cols, index=val_df.index)
test_scaled = pd.DataFrame(scaler.transform(test_df), columns=feature_cols, index=test_df.index)

train_scaled.describe()

,PM10,PM2.5,Carbon Monoxide,Nitrogen Dioxide,Sulphur Dioxide,Ozone,Aerosol Optical Depth,hour_sin,hour_cos,month_sin,month_cos
count,24405.000000,24405.000000,24405.000000,24405.000000,24405.000000,24405.000000,24405.000000,24405.000000,24405.000000,24405.000000,24405.000000
mean,0.213732,0.189228,0.195096,0.106943,0.153620,0.278850,0.101510,0.499942,0.499984,0.502905,0.534046
std,0.104979,0.096643,0.102373,0.086840,0.107413,0.115454,0.075255,0.353543,0.353578,0.361673,0.343562
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.139630,0.124118,0.122087,0.051502,0.082090,0.189516,0.055319,0.146447,0.146447,0.066987,0.250000
50%,0.198152,0.173484,0.174251,0.081545,0.126866,0.270161,0.085106,0.500000,0.500000,0.500000,0.500000
75%,0.274127,0.238364,0.247503,0.133047,0.201493,0.354839,0.123404,0.853553,0.853553,0.933013,0.933013
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


##### Lưu dữ liệu đã xử lý để notebook khác (04_modeling.ipynb) dùng lại

In [29]:
import os
import joblib

# Lưu train/val/test đã scale + scaler ra data/processed/ để 04_modeling.ipynb import lại thay vì
# phải lặp lại toàn bộ pipeline preprocessing -> tránh code trùng lặp, đảm bảo 2 notebook luôn đồng bộ dữ liệu
os.makedirs('data/processed', exist_ok=True)

train_scaled.to_csv('data/processed/train_scaled.csv')
val_scaled.to_csv('data/processed/val_scaled.csv')
test_scaled.to_csv('data/processed/test_scaled.csv')
joblib.dump(scaler, 'data/processed/scaler.joblib')

print("Đã lưu train_scaled.csv, val_scaled.csv, test_scaled.csv, scaler.joblib vào data/processed/")

Đã lưu train_scaled.csv, val_scaled.csv, test_scaled.csv, scaler.joblib vào data/processed/


##### Bước 6: Sliding Window (tạo cặp X, y)

In [30]:
def create_sequences(data, target_col_idx, window_size, horizon=1):
    X, y = [], []
    for i in range(len(data) - window_size - horizon + 1):
        X.append(data[i:i + window_size])
        y.append(data[i + window_size + horizon - 1, target_col_idx])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


WINDOW_SIZE = 48  # giờ (~2 ngày) - dựa theo chu kỳ ngày-đêm đã xác nhận qua ACF/PACF & Periodogram (02_eda.ipynb)
HORIZON = 1       # dự báo 1 giờ tới

X_train, y_train = create_sequences(train_scaled.values, pm25_col_idx, WINDOW_SIZE, HORIZON)
X_val, y_val = create_sequences(val_scaled.values, pm25_col_idx, WINDOW_SIZE, HORIZON)
X_test, y_test = create_sequences(test_scaled.values, pm25_col_idx, WINDOW_SIZE, HORIZON)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test:  {X_test.shape}, y_test: {y_test.shape}")

X_train: (24357, 48, 11), y_train: (24357,)
X_val:   (5182, 48, 11), y_val: (5182,)
X_test:  (5182, 48, 11), y_test: (5182,)


##### Bước 7: Thiết kế kiến trúc GRU (PyTorch)

In [31]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 64
train_loader = DataLoader(TimeSeriesDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TimeSeriesDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TimeSeriesDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

In [32]:
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)     # out: [batch, window, hidden]
        out = out[:, -1, :]      # lấy hidden state ở bước thời gian cuối cùng của sequence
        return self.fc(out).squeeze(-1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GRUModel(input_size=X_train.shape[2]).to(device)
print(f"Device: {device}")
print(model)

Device: cpu
GRUModel(
  (gru): GRU(11, 64, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [33]:
# Sanity check: forward pass thử với 1 batch để đảm bảo shape khớp trước khi sang bước training ở notebook sau
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Tổng số tham số có thể train: {n_params:,}")

xb, yb = next(iter(train_loader))
xb = xb.to(device)
with torch.no_grad():
    pred = model(xb)
print(f"Input batch shape: {tuple(xb.shape)} -> Output shape: {tuple(pred.shape)}")

Tổng số tham số có thể train: 39,809
Input batch shape: (64, 48, 11) -> Output shape: (64,)
